# 06 - Final Model Selection

This notebook selects the final fraud detection model for the project.

Reasoning: model selection is not just choosing the highest ROC-AUC. In fraud detection, the final decision must balance missed fraud, false alerts, ranking quality, threshold behavior, and overfitting risk.

## 1. Setup

Reasoning: this notebook consumes the evaluation outputs. If `05_model_evaluation.ipynb` has not been run yet, it can still fall back to the final test metrics from `04_model_tuning.ipynb`, but the preferred flow is to run evaluation first.

In [ ]:
from pathlib import Path
import json
import pickle
import shutil
import warnings

import numpy as np
import pandas as pd

from IPython.display import display, Markdown

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

REPORTS_PATH = PROJECT_ROOT / 'reports'
MODELS_PATH = PROJECT_ROOT / 'models'

REPORTS_PATH.mkdir(exist_ok=True)
MODELS_PATH.mkdir(exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')

## 2. Load Evidence From the Project

Reasoning: the final decision should be traceable back to the whole project: EDA showed the class imbalance, preprocessing protected against leakage, baseline training showed model families, tuning improved thresholds, and evaluation measured final behavior.

In [ ]:
summary_files = [
    REPORTS_PATH / 'eda_summary.txt',
    REPORTS_PATH / 'preprocessing_summary.txt',
    REPORTS_PATH / 'training_summary.txt',
    REPORTS_PATH / 'tuning_summary.txt',
    REPORTS_PATH / 'final_evaluation_summary.txt',
]

for summary_file in summary_files:
    if summary_file.exists():
        display(Markdown(f'### {summary_file.name}'))
        text = summary_file.read_text(encoding='utf-8', errors='replace')
        print(text[:1800].encode('ascii', errors='replace').decode('ascii'))
        if len(text) > 1800:
            print('\n... summary truncated for display ...')
    else:
        print(f'Not found yet: {summary_file.name}')

baseline_path = REPORTS_PATH / 'baseline_model_comparison.csv'
tuning_path = REPORTS_PATH / 'tuning_results.csv'
evaluation_path = REPORTS_PATH / 'final_evaluation_results.csv'

baseline_df = pd.read_csv(baseline_path) if baseline_path.exists() else pd.DataFrame()
tuning_df = pd.read_csv(tuning_path) if tuning_path.exists() else pd.DataFrame()

if evaluation_path.exists():
    evaluation_df = pd.read_csv(evaluation_path)
    evidence_source = 'reports/final_evaluation_results.csv'
elif not tuning_df.empty:
    evaluation_df = tuning_df.rename(columns={
        'Validation Threshold': 'Threshold',
        'Test Precision @Tuned': 'Precision',
        'Test Recall @Tuned': 'Recall',
        'Test F1 @Tuned': 'F1',
        'Test F2 @Tuned': 'F2',
        'Test ROC-AUC': 'ROC-AUC',
        'Test Average Precision': 'Average Precision',
        'Test FP @Tuned': 'FP',
        'Test FN @Tuned': 'FN',
        'Test TP @Tuned': 'TP',
    }).copy()
    evidence_source = 'reports/tuning_results.csv fallback'
else:
    raise FileNotFoundError('Run notebooks/04_model_tuning.ipynb, then notebooks/05_model_evaluation.ipynb before selection.')

print(f'Using model evidence from: {evidence_source}')
display(Markdown('### Baseline Comparison'))
display(baseline_df)
display(Markdown('### Final Evaluation Evidence'))
display(evaluation_df)

## 3. Define Selection Gates

Reasoning: gates are minimum requirements. A model can have a high score but still be unacceptable if it misses too much fraud, creates too many false alarms, or shows too much overfitting. These gates are intentionally practical for this dataset and can be adjusted for a different business policy.

In [ ]:
SELECTION_GATES = {
    'Minimum Precision': 0.80,
    'Minimum Recall': 0.75,
    'Minimum F2': 0.75,
    'Minimum Average Precision': 0.75,
    'Maximum Overfit Gap': 0.06,
    'Maximum False Positives': 10,
}

for gate, value in SELECTION_GATES.items():
    print(f'{gate}: {value}')

gate_reasoning = """
Gate reasoning:
- Precision >= 0.80 keeps the review queue useful. A low precision fraud model can overwhelm investigators.
- Recall >= 0.75 keeps the model focused on catching fraud. Missing too many fraud cases defeats the project goal.
- F2 >= 0.75 rewards recall more than precision while still requiring both to be meaningful.
- Average precision >= 0.75 checks that fraud cases are ranked near the top under severe class imbalance.
- Overfit gap <= 0.06 avoids choosing a model that memorizes training data but generalizes poorly.
- False positives <= 10 on this test split keeps the operational workload small.
"""
print(gate_reasoning)

## 4. Score the Candidate Models

Reasoning: after gates, a weighted score ranks the acceptable models. The highest weights go to F2 and average precision because this project needs both good fraud capture and strong fraud ranking.

In [ ]:
WEIGHTS = {
    'F2': 0.25,
    'Average Precision': 0.20,
    'Precision': 0.15,
    'Recall': 0.15,
    'ROC-AUC': 0.10,
    'Low FP': 0.05,
    'Low FN': 0.05,
    'Stability': 0.05,
}

def normalize(series, higher_is_better=True):
    values = series.astype(float)
    if values.max() == values.min():
        normalized = pd.Series(1.0, index=values.index)
    else:
        normalized = (values - values.min()) / (values.max() - values.min())
    if not higher_is_better:
        normalized = 1 - normalized
    return normalized

selection_df = evaluation_df.copy()

for column in ['Precision', 'Recall', 'F1', 'F2', 'ROC-AUC', 'Average Precision', 'FP', 'FN', 'Overfit Gap']:
    if column not in selection_df.columns:
        selection_df[column] = 0

selection_df['Pass Precision Gate'] = selection_df['Precision'] >= SELECTION_GATES['Minimum Precision']
selection_df['Pass Recall Gate'] = selection_df['Recall'] >= SELECTION_GATES['Minimum Recall']
selection_df['Pass F2 Gate'] = selection_df['F2'] >= SELECTION_GATES['Minimum F2']
selection_df['Pass AP Gate'] = selection_df['Average Precision'] >= SELECTION_GATES['Minimum Average Precision']
selection_df['Pass Overfit Gate'] = selection_df['Overfit Gap'] <= SELECTION_GATES['Maximum Overfit Gap']
selection_df['Pass FP Gate'] = selection_df['FP'] <= SELECTION_GATES['Maximum False Positives']

gate_columns = [
    'Pass Precision Gate',
    'Pass Recall Gate',
    'Pass F2 Gate',
    'Pass AP Gate',
    'Pass Overfit Gate',
    'Pass FP Gate',
]
selection_df['Eligible'] = selection_df[gate_columns].all(axis=1)

selection_df['Score F2'] = normalize(selection_df['F2'])
selection_df['Score AP'] = normalize(selection_df['Average Precision'])
selection_df['Score Precision'] = normalize(selection_df['Precision'])
selection_df['Score Recall'] = normalize(selection_df['Recall'])
selection_df['Score ROC-AUC'] = normalize(selection_df['ROC-AUC'])
selection_df['Score Low FP'] = normalize(selection_df['FP'], higher_is_better=False)
selection_df['Score Low FN'] = normalize(selection_df['FN'], higher_is_better=False)
selection_df['Score Stability'] = normalize(selection_df['Overfit Gap'], higher_is_better=False)

selection_df['Selection Score'] = (
    WEIGHTS['F2'] * selection_df['Score F2']
    + WEIGHTS['Average Precision'] * selection_df['Score AP']
    + WEIGHTS['Precision'] * selection_df['Score Precision']
    + WEIGHTS['Recall'] * selection_df['Score Recall']
    + WEIGHTS['ROC-AUC'] * selection_df['Score ROC-AUC']
    + WEIGHTS['Low FP'] * selection_df['Score Low FP']
    + WEIGHTS['Low FN'] * selection_df['Score Low FN']
    + WEIGHTS['Stability'] * selection_df['Score Stability']
)

eligible_df = selection_df[selection_df['Eligible']].copy()
if eligible_df.empty:
    print('No model passed every gate. Selecting the highest score while flagging the gate failure.')
    selected_row = selection_df.sort_values('Selection Score', ascending=False).iloc[0]
else:
    selected_row = eligible_df.sort_values('Selection Score', ascending=False).iloc[0]

selection_df = selection_df.sort_values(['Eligible', 'Selection Score'], ascending=False).reset_index(drop=True)

selection_path = REPORTS_PATH / 'final_model_decision.csv'
selection_df.to_csv(selection_path, index=False)

display(selection_df)
print(f'Selected model: {selected_row["Model"]}')
print(f'Saved: {selection_path}')

## 5. Explain the Final Choice

Reasoning: the selected model must be explainable in plain language. This section writes down why it won, why the other models were not selected, and what threshold should be used.

In [ ]:
selected_model_name = selected_row['Model']
selected_threshold = float(selected_row['Threshold']) if 'Threshold' in selected_row.index else None

rejection_reasons = []
for _, row in selection_df.iterrows():
    if row['Model'] == selected_model_name:
        continue
    failed = [col.replace('Pass ', '').replace(' Gate', '') for col in gate_columns if not row[col]]
    if failed:
        reason = f"failed gate(s): {', '.join(failed)}"
    else:
        reason = f"passed gates but had lower selection score ({row['Selection Score']:.4f})"
    rejection_reasons.append((row['Model'], reason))

print(f'Final model: {selected_model_name}')
print(f'Final threshold: {selected_threshold:.5f}')
print('\nWhy other models were not selected:')
for model_name, reason in rejection_reasons:
    print(f'- {model_name}: {reason}')

## 6. Save Final Model Alias and Metadata

Reasoning: downstream users should not need to know which tuning filename won. The selected model is copied to `models/final_model.pkl`, and metadata stores the threshold and evaluation metrics needed for consistent predictions.

In [ ]:
if 'Saved Model' in selected_row.index and pd.notna(selected_row['Saved Model']):
    selected_model_path = PROJECT_ROOT / str(selected_row['Saved Model']).replace('\\', '/')
elif not tuning_df.empty and selected_model_name in tuning_df['Model'].values:
    selected_model_path = PROJECT_ROOT / str(tuning_df.loc[tuning_df['Model'] == selected_model_name, 'Saved Model'].iloc[0]).replace('\\', '/')
else:
    raise FileNotFoundError('Could not find the saved model path for the selected model.')

final_model_path = MODELS_PATH / 'final_model.pkl'
shutil.copyfile(selected_model_path, final_model_path)

metadata = {
    'selected_model': selected_model_name,
    'source_model_file': str(selected_model_path.relative_to(PROJECT_ROOT)),
    'final_model_file': str(final_model_path.relative_to(PROJECT_ROOT)),
    'threshold': selected_threshold,
    'selection_score': float(selected_row['Selection Score']),
    'eligible': bool(selected_row['Eligible']),
    'metrics': {
        'precision': float(selected_row['Precision']),
        'recall': float(selected_row['Recall']),
        'f1': float(selected_row['F1']),
        'f2': float(selected_row['F2']),
        'roc_auc': float(selected_row['ROC-AUC']),
        'average_precision': float(selected_row['Average Precision']),
        'false_positives': int(selected_row['FP']),
        'false_negatives': int(selected_row['FN']),
        'true_positives': int(selected_row['TP']) if 'TP' in selected_row.index else None,
        'overfit_gap': float(selected_row['Overfit Gap']),
    },
    'selection_gates': SELECTION_GATES,
    'selection_weights': WEIGHTS,
}

metadata_path = MODELS_PATH / 'final_model_metadata.json'
metadata_path.write_text(json.dumps(metadata, indent=2), encoding='utf-8')

print(f'Copied selected model to: {final_model_path}')
print(f'Saved metadata to: {metadata_path}')

## 7. Final Model Card and Selection Report

Reasoning: a project is easier to defend when the final choice is documented with intended use, assumptions, limitations, and the exact metrics that justify the decision.

In [ ]:
model_card = f"""
# Final Fraud Detection Model Card

## Selected Model

- Model: {selected_model_name}
- Saved file: `models/final_model.pkl`
- Decision threshold: {selected_threshold:.5f}
- Selection score: {selected_row['Selection Score']:.4f}

## Why This Model Was Selected

This model was selected because it passed the required gates and had the best weighted balance of fraud-catching ability, precision, ranking quality, and generalization stability.

The most important project constraint is the class imbalance: fraud cases are extremely rare, so accuracy is not meaningful as a decision metric. The selected model is judged by precision, recall, F2, ROC-AUC, average precision, false positives, false negatives, and overfitting gap.

## Final Test Metrics

- Precision: {selected_row['Precision']:.4f}
- Recall: {selected_row['Recall']:.4f}
- F1: {selected_row['F1']:.4f}
- F2: {selected_row['F2']:.4f}
- ROC-AUC: {selected_row['ROC-AUC']:.4f}
- Average precision: {selected_row['Average Precision']:.4f}
- False positives: {int(selected_row['FP'])}
- False negatives: {int(selected_row['FN'])}
- Overfit gap: {selected_row['Overfit Gap']:.4f}

## Intended Use

The model is intended to rank credit card transactions by fraud risk and flag transactions above the selected threshold for review or downstream action.

## Preprocessing Assumptions

- Use the same preprocessing flow from `02_preprocessing.ipynb`.
- Use the saved scaler and feature order from the processed training data.
- Do not fit preprocessing steps on future test or production data.
- Keep raw and processed data local; the `data/` folder is ignored by Git.

## Limitations

- The dataset is highly imbalanced, so small changes in threshold can materially change false positives and false negatives.
- The model was evaluated on historical data and should be monitored for drift.
- The selected threshold reflects the current balance between catching fraud and limiting review workload. A different business cost policy may choose a different threshold.

## Monitoring Recommendation

Track precision, recall, false positives, false negatives, fraud base rate, score distribution, and feature drift over time. Re-tune the threshold if the fraud base rate or investigation capacity changes.
""".strip()

selection_summary_lines = [
    '=' * 80,
    'FINAL MODEL SELECTION REPORT',
    '=' * 80,
    '',
    '1. DECISION PRINCIPLE',
    '   The selected model must catch fraud reliably without creating an unreasonable review workload.',
    '   Accuracy is not used because the dataset is extremely imbalanced.',
    '',
    '2. SELECTED MODEL',
    f"   - Model: {selected_model_name}",
    f"   - Threshold: {selected_threshold:.5f}",
    f"   - Selection score: {selected_row['Selection Score']:.4f}",
    f"   - Eligible by gates: {bool(selected_row['Eligible'])}",
    '',
    '3. FINAL TEST PERFORMANCE',
    f"   - Precision: {selected_row['Precision']:.4f}",
    f"   - Recall: {selected_row['Recall']:.4f}",
    f"   - F1: {selected_row['F1']:.4f}",
    f"   - F2: {selected_row['F2']:.4f}",
    f"   - ROC-AUC: {selected_row['ROC-AUC']:.4f}",
    f"   - Average precision: {selected_row['Average Precision']:.4f}",
    f"   - False positives: {int(selected_row['FP'])}",
    f"   - False negatives: {int(selected_row['FN'])}",
    '',
    '4. WHY OTHER MODELS WERE NOT SELECTED',
]

for model_name, reason in rejection_reasons:
    selection_summary_lines.append(f'   - {model_name}: {reason}')

selection_summary_lines.extend([
    '',
    '5. FILES CREATED',
    '   - reports/final_model_decision.csv',
    '   - reports/model_selection_summary.txt',
    '   - reports/final_model_card.md',
    '   - models/final_model.pkl',
    '   - models/final_model_metadata.json',
    '',
    '6. FINAL PROJECT NEXT STEP',
    '   Use the final model only with the documented preprocessing pipeline and threshold.',
    '   For explanation, continue with SHAP analysis on the selected final model.',
    '',
    '=' * 80,
])

selection_summary = '\n'.join(selection_summary_lines)

model_card_path = REPORTS_PATH / 'final_model_card.md'
summary_path = REPORTS_PATH / 'model_selection_summary.txt'
model_card_path.write_text(model_card, encoding='utf-8')
summary_path.write_text(selection_summary, encoding='utf-8')

display(Markdown(model_card))
print(selection_summary)
print(f'Saved: {model_card_path}')
print(f'Saved: {summary_path}')